# OCR keyframes bằng PP-OCRv5 và VietOCR

Notebook độc lập cho Kaggle. Bật **GPU** và **Internet**. **Lần đầu bấm Run all**, notebook cài dependency rồi tự khởi động lại kernel; đợi kernel sẵn sàng và **bấm Run all lần hai** để OCR. Paddle GPU được cài không kèm NVIDIA dependency để giữ nguyên NCCL của Kaggle/PyTorch.

In [ ]:
from pathlib import Path

EXPECTED_VIDEO_COUNT = 873
MAX_VIDEOS_PER_RUN = 20
VIDEO_PREFIXES = ()  # Ví dụ ("L21",); để rỗng để xử lý toàn bộ batch 1.

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/ocr")
RECORD_ROOT = OUTPUT_ROOT / "records"


In [ ]:
import os
import signal
import subprocess
import sys
import time
from pathlib import Path

install_marker = Path("/kaggle/working/.ocr_dependencies_v4_ready")
if not install_marker.exists():
    # Gỡ runtime xung đột và LangChain tùy chọn bị cài thiếu trong image Kaggle.
    subprocess.run([
        sys.executable, "-m", "pip", "uninstall", "-y",
        "paddlepaddle", "paddlepaddle-gpu", "langchain", "langchain-community",
    ], check=False)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "paddleocr==3.5.0", "vietocr==0.3.13",
    ], check=True)
    # --no-deps tránh Paddle hạ phiên bản NCCL/cuDNN đang được PyTorch Kaggle dùng.
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "--no-deps",
        "--index-url", "https://www.paddlepaddle.org.cn/packages/stable/cu126/",
        "paddlepaddle-gpu==3.3.0",
    ], check=True)
    install_marker.write_text("ready", encoding="utf-8")
    print("Đã cài xong. Kernel đang restart; sau đó bấm Run all lần hai.", flush=True)
    time.sleep(1)
    os.kill(os.getpid(), signal.SIGKILL)
print("Dependency đã sẵn sàng; không chạy lại pip.")


In [ ]:
import csv
import hashlib
import json
import os
import re
import shutil

import cv2
import numpy as np
import torch
import paddle
from PIL import Image

os.environ["PADDLE_PDX_EAGER_INIT"] = "0"
from paddleocr import PaddleOCR
from vietocr.tool.config import Cfg
from vietocr.tool.predictor import Predictor

if not torch.cuda.is_available():
    raise RuntimeError("PyTorch không thấy GPU")
if not paddle.is_compiled_with_cuda() or paddle.device.cuda.device_count() == 0:
    raise RuntimeError("Paddle không thấy GPU")
print(f"Paddle={paddle.__version__} Torch={torch.__version__} GPU={torch.cuda.get_device_name(0)}")


In [ ]:
VIDEO_PATTERN = re.compile(r"L\d+_V\d+")
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}

def find_keyframe_directories(roots):
    found = {}
    for root in roots:
        for directory in root.glob("**/L*_V*"):
            if not directory.is_dir() or not VIDEO_PATTERN.fullmatch(directory.name):
                continue
            if any(path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES for path in directory.iterdir()):
                found[directory.name] = directory  # Version mới hơn thắng nếu dataset có snapshot lặp.
    return found

def find_mapping_files(roots):
    found = {}
    for root in roots:
        for path in root.glob("**/L*_V*.csv"):
            if VIDEO_PATTERN.fullmatch(path.stem):
                found[path.stem] = path
    return found

# Kaggle có thể mount dataset bằng slug hoặc đường dẫn versions/<n>; quét trực tiếp input đã gắn.
keyframe_sources = find_keyframe_directories([KAGGLE_INPUT_ROOT])
mapping_sources = find_mapping_files([KAGGLE_INPUT_ROOT])
if len(keyframe_sources) != EXPECTED_VIDEO_COUNT:
    raise ValueError(f"Cần {EXPECTED_VIDEO_COUNT} video, tìm thấy {len(keyframe_sources)}")
missing_mappings = sorted(set(keyframe_sources) - set(mapping_sources))
if missing_mappings:
    raise ValueError(f"Thiếu mapping cho {len(missing_mappings)} video: {missing_mappings[:5]}")

RECORD_ROOT.mkdir(parents=True, exist_ok=True)
for previous in KAGGLE_INPUT_ROOT.glob("**/records/L*_V*.jsonl"):
    destination = RECORD_ROOT / previous.name
    if not destination.exists():
        shutil.copy2(previous, destination)

video_ids = sorted(keyframe_sources)
if VIDEO_PREFIXES:
    video_ids = [video_id for video_id in video_ids if video_id.startswith(VIDEO_PREFIXES)]
pending = [video_id for video_id in video_ids if not (RECORD_ROOT / f"{video_id}.jsonl").exists()]
run_video_ids = pending[:MAX_VIDEOS_PER_RUN]
print(f"videos={len(video_ids)} completed={len(video_ids) - len(pending)} run={len(run_video_ids)} remaining={len(pending)}")


In [ ]:
ppocr = PaddleOCR(
    text_detection_model_name="PP-OCRv5_mobile_det",
    text_recognition_model_name="latin_PP-OCRv5_mobile_rec",
    text_recognition_batch_size=16,
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    device="gpu:0",
    engine="paddle",
)

vietocr_config = Cfg.load_config_from_name("vgg_seq2seq")
vietocr_config["device"] = "cuda:0"
vietocr_config["cnn"]["pretrained"] = False
vietocr_config["predictor"]["beamsearch"] = False
vietocr_weights_url = vietocr_config["weights"]
vietocr = Predictor(vietocr_config)


In [ ]:
def mapping_value(row, names, required=True):
    value = next((row.get(name) for name in names if row.get(name) not in (None, "")), None)
    if value is None and required:
        raise ValueError(f"Thiếu một trong các cột {names}")
    return value

def read_mapping(path):
    with path.open(newline="", encoding="utf-8-sig") as file:
        rows = list(csv.DictReader(file))
    return [{
        "frame_id": int(float(mapping_value(row, ("frame_id", "frame_idx", "frame")))),
        "timestamp": float(value) if (value := mapping_value(row, ("timestamp", "pts_time", "time"), False)) is not None else None,
    } for row in rows]

def perspective_crop(rgb_image, polygon):
    points = np.asarray(polygon, dtype=np.float32).reshape(4, 2)
    ordered = np.empty((4, 2), dtype=np.float32)
    ordered[0], ordered[2] = points[np.argmin(points.sum(1))], points[np.argmax(points.sum(1))]
    ordered[1], ordered[3] = points[np.argmin(np.diff(points, axis=1))], points[np.argmax(np.diff(points, axis=1))]
    width = max(1, round(max(np.linalg.norm(ordered[1] - ordered[0]), np.linalg.norm(ordered[2] - ordered[3]))))
    height = max(1, round(max(np.linalg.norm(ordered[3] - ordered[0]), np.linalg.norm(ordered[2] - ordered[1]))))
    target = np.float32([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]])
    crop = cv2.warpPerspective(rgb_image, cv2.getPerspectiveTransform(ordered, target), (width, height))
    if height > width * 1.5:
        crop = np.rot90(crop)
    return Image.fromarray(crop)

def unique_search_text(detections):
    values, seen = [], set()
    for detection in detections:
        for key in ("ppocr_text", "vietocr_text"):
            text = " ".join(detection[key].split())
            normalized = text.casefold()
            if text and normalized not in seen:
                seen.add(normalized)
                values.append(text)
    return " \n".join(values)

demo = np.zeros((20, 40, 3), dtype=np.uint8)
assert perspective_crop(demo, [[0, 0], [39, 0], [39, 19], [0, 19]]).size == (39, 19)


In [ ]:
for video_position, video_id in enumerate(run_video_ids, 1):
    image_paths = sorted(path for path in keyframe_sources[video_id].iterdir() if path.suffix.lower() in IMAGE_SUFFIXES)
    mappings = read_mapping(mapping_sources[video_id])
    if len(image_paths) != len(mappings):
        raise ValueError(f"{video_id}: keyframes={len(image_paths)} mappings={len(mappings)}")

    output_path = RECORD_ROOT / f"{video_id}.jsonl"
    temporary_path = output_path.with_suffix(".jsonl.tmp")
    temporary_path.unlink(missing_ok=True)
    detection_count = 0
    with temporary_path.open("w", encoding="utf-8") as output:
        for image_path, mapping in zip(image_paths, mappings):
            result = next(iter(ppocr.predict(str(image_path), text_rec_score_thresh=0.0))).json
            data = result.get("res", result)
            polygons = data.get("rec_polys", [])
            ppocr_texts = data.get("rec_texts", [])
            ppocr_scores = data.get("rec_scores", [])
            detection_scores = data.get("dt_scores", [])
            if not (len(polygons) == len(ppocr_texts) == len(ppocr_scores)):
                raise ValueError(f"Kết quả PP-OCR lệch cột: {image_path}")

            with Image.open(image_path) as image:
                rgb_image = np.asarray(image.convert("RGB"))
            crops = [perspective_crop(rgb_image, polygon) for polygon in polygons]
            if crops:
                vietocr_texts, vietocr_scores = vietocr.predict_batch(crops, return_prob=True)
            else:
                vietocr_texts, vietocr_scores = [], []

            detections = []
            for index, (polygon, paddle_text, paddle_score, viet_text, viet_score) in enumerate(zip(
                polygons, ppocr_texts, ppocr_scores, vietocr_texts, vietocr_scores
            )):
                detections.append({
                    "box": np.asarray(polygon).astype(int).tolist(),
                    "det_score": float(detection_scores[index]) if len(detection_scores) == len(polygons) else None,
                    "ppocr_text": str(paddle_text),
                    "ppocr_score": float(paddle_score),
                    "vietocr_text": str(viet_text),
                    "vietocr_score": float(viet_score),
                })
            record = {
                "video_id": video_id,
                "keyframe_id": image_path.stem,
                "frame_id": mapping["frame_id"],
                "timestamp": mapping["timestamp"],
                "image_path": f"keyframes/{video_id}/{image_path.name}",
                "search_text": unique_search_text(detections),
                "detections": detections,
            }
            output.write(json.dumps(record, ensure_ascii=False) + "\n")
            detection_count += len(detections)
    temporary_path.replace(output_path)
    print(f"[{video_position}/{len(run_video_ids)}] {video_id}: frames={len(image_paths)} detections={detection_count}", flush=True)


In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

reports, missing = [], []
for video_id in video_ids:
    path = RECORD_ROOT / f"{video_id}.jsonl"
    if not path.exists():
        missing.append(video_id)
        continue
    with path.open(encoding="utf-8") as file:
        records = [json.loads(line) for line in file if line.strip()]
    expected = sum(1 for path in keyframe_sources[video_id].iterdir() if path.suffix.lower() in IMAGE_SUFFIXES)
    if len(records) != expected or any(record["video_id"] != video_id for record in records):
        raise ValueError(f"Output không hợp lệ: {video_id}")
    reports.append({"video_id": video_id, "keyframes": len(records), "sha256": file_sha256(path)})

weight_name = vietocr_weights_url.rsplit("/", 1)[-1]
weight_path = Path("/tmp") / weight_name
manifest = {
    "schema_version": 1,
    "ppocr_detection_model": "PP-OCRv5_mobile_det",
    "ppocr_recognition_model": "latin_PP-OCRv5_mobile_rec",
    "vietocr_model": "vgg_seq2seq",
    "vietocr_weights_url": vietocr_weights_url,
    "vietocr_weights_sha256": file_sha256(weight_path) if weight_path.exists() else None,
    "videos": reports,
    "missing_video_ids": missing,
}
(OUTPUT_ROOT / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
archive = shutil.make_archive("/kaggle/working/ocr-results", "zip", OUTPUT_ROOT)
print(f"completed={len(reports)} missing={len(missing)} download={archive}")


In [ ]:
completed_paths = sorted(RECORD_ROOT.glob("L*_V*.jsonl"))
if completed_paths:
    with completed_paths[-1].open(encoding="utf-8") as file:
        sample = json.loads(next(file))
    print(json.dumps(sample, ensure_ascii=False, indent=2)[:5000])
